# Real-Time Market Microstructure Analysis

This notebook demonstrates real-time analysis of cryptocurrency markets using the streaming platform.

**Features:**
- Live WebSocket streaming from Binance
- Order flow analysis (buy/sell pressure)
- Liquidity metrics
- Volume profile and VWAP
- Real-time visualizations

**Note:** Run cells sequentially. The consumer will run for a specified duration, then you can analyze the collected metrics.

## Setup and Imports

In [1]:
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from binance_tick_data.consumers import RealtimeConsumer, ConsumerConfig
from binance_tick_data.analyzers import (
    OrderFlowAnalyzer,
    LiquidityAnalyzer,
    VolumeProfileAnalyzer,
)
from binance_tick_data.streaming import MetricsAggregator

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Imports successful")

NameError: name 'Dict' is not defined

## Configuration

In [ ]:
# Streaming configuration
SYMBOL = "BTCUSDT"
DURATION_SECONDS = 120  # Run for 2 minutes
WINDOW_SIZE = 60  # 60 second analysis windows
BUFFER_SIZE = 10000

print(f"Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Duration: {DURATION_SECONDS}s")
print(f"  Window size: {WINDOW_SIZE}s")

## Initialize Consumer and Analyzers

In [ ]:
# Create consumer configuration
config = ConsumerConfig(
    symbols=[SYMBOL],
    buffer_size=BUFFER_SIZE,
    update_interval=0.1,
    reconnect_delay=5,
)

# Create consumer
consumer = RealtimeConsumer(config)

# Create analyzers
order_flow = OrderFlowAnalyzer(
    window_size=WINDOW_SIZE,
    compute_toxicity=True,
    persistence_test=True,
)

liquidity = LiquidityAnalyzer(
    window_size=WINDOW_SIZE,
    spread_method="mid",
)

volume_profile = VolumeProfileAnalyzer(
    window_size=WINDOW_SIZE,
    price_bins=50,
    vwap_decay=0.99,
)

# Register analyzers
consumer.register_analyzer(order_flow)
consumer.register_analyzer(liquidity)
consumer.register_analyzer(volume_profile)

# Create metrics aggregator
aggregator = MetricsAggregator(
    window_sizes=[10, 30, 60],
    publish_interval=1.0,
)

print("✓ Consumer and analyzers initialized")

## Collect Real-Time Data

This cell will run the consumer for the specified duration and collect metrics.

In [ ]:
# Storage for metrics history
metrics_history = []

async def collect_data():
    """Collect streaming data for specified duration."""
    print(f"Starting data collection for {DURATION_SECONDS} seconds...")
    print("This may take a moment to establish connection...\n")
    
    # Start consumer
    await consumer.start()
    print("✓ Consumer started, receiving trades...\n")
    
    start_time = datetime.now()
    
    # Collect metrics every 5 seconds
    try:
        while (datetime.now() - start_time).total_seconds() < DURATION_SECONDS:
            await asyncio.sleep(5)
            
            # Get current metrics from all analyzers
            timestamp = datetime.now()
            snapshot = {'timestamp': timestamp}
            
            # Order flow metrics
            of_metrics = order_flow.get_current_metrics()
            for key, value in of_metrics.items():
                snapshot[f'order_flow_{key}'] = value
                if isinstance(value, (int, float)) and not np.isinf(value):
                    aggregator.add_metric(f'order_flow_{key}', value, timestamp)
            
            # Liquidity metrics
            liq_metrics = liquidity.get_current_metrics()
            for key, value in liq_metrics.items():
                snapshot[f'liquidity_{key}'] = value
                if isinstance(value, (int, float)) and not np.isinf(value):
                    aggregator.add_metric(f'liquidity_{key}', value, timestamp)
            
            # Volume profile metrics
            vp_metrics = volume_profile.get_current_metrics()
            for key, value in vp_metrics.items():
                snapshot[f'volume_profile_{key}'] = value
                if isinstance(value, (int, float)) and not np.isinf(value):
                    aggregator.add_metric(f'volume_profile_{key}', value, timestamp)
            
            metrics_history.append(snapshot)
            
            elapsed = (datetime.now() - start_time).total_seconds()
            print(f"[{elapsed:.0f}s] Collected snapshot - "
                  f"Trades: {snapshot.get('order_flow_total_trades', 0)}, "
                  f"VWAP: {snapshot.get('volume_profile_vwap', 0):.2f}")
    
    finally:
        # Stop consumer
        await consumer.stop()
        print("\n✓ Data collection complete!")

# Run the collection
await collect_data()

# Convert to DataFrame
df_metrics = pd.DataFrame(metrics_history)
print(f"\nCollected {len(df_metrics)} metric snapshots")
print(f"Metrics tracked: {len(df_metrics.columns) - 1}")  # -1 for timestamp

## Summary Statistics

In [ ]:
# Get final statistics
stats = consumer.get_statistics()

print("=" * 60)
print("COLLECTION SUMMARY")
print("=" * 60)
print(f"Duration: {stats.get('uptime_seconds', 0):.1f} seconds")
print(f"Symbol: {SYMBOL}")
print(f"Analyzers: {', '.join(stats['analyzers'])}")

if SYMBOL in stats['symbols_data']:
    symbol_stats = stats['symbols_data'][SYMBOL]
    print(f"\n{SYMBOL} Statistics:")
    print(f"  Total trades: {symbol_stats['total_trades']:,}")
    print(f"  Buffer utilization: {symbol_stats['utilization']:.1%}")
    print(f"  Trades in buffer: {symbol_stats['current_size']:,}")

## Visualization 1: Order Flow Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Trade counts
if 'order_flow_buy_count' in df_metrics.columns:
    axes[0].plot(df_metrics['timestamp'], df_metrics['order_flow_buy_count'], 
                 label='Buy Trades', color='green', linewidth=2)
    axes[0].plot(df_metrics['timestamp'], df_metrics['order_flow_sell_count'], 
                 label='Sell Trades', color='red', linewidth=2)
    axes[0].set_title('Order Flow: Buy vs Sell Trade Counts', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Trade Count', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Order imbalance
if 'order_flow_order_imbalance' in df_metrics.columns:
    axes[1].plot(df_metrics['timestamp'], df_metrics['order_flow_order_imbalance'], 
                 label='Order Imbalance', color='blue', linewidth=2)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1].axhline(y=0.1, color='green', linestyle=':', alpha=0.3, label='Threshold')
    axes[1].axhline(y=-0.1, color='red', linestyle=':', alpha=0.3)
    axes[1].set_title('Order Imbalance (Buy - Sell) / Total', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Imbalance', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Buy/sell ratio
if 'order_flow_buy_sell_ratio' in df_metrics.columns:
    # Cap ratio for visualization
    ratio = df_metrics['order_flow_buy_sell_ratio'].clip(upper=3.0)
    axes[2].plot(df_metrics['timestamp'], ratio, 
                 label='Buy/Sell Ratio', color='purple', linewidth=2)
    axes[2].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Neutral')
    axes[2].set_title('Buy/Sell Ratio (capped at 3.0)', fontsize=14, fontweight='bold')
    axes[2].set_ylabel('Ratio', fontsize=12)
    axes[2].set_xlabel('Time', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Visualization 2: Volume Profile and VWAP

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# VWAP over time
if 'volume_profile_vwap' in df_metrics.columns:
    vwap = df_metrics['volume_profile_vwap']
    axes[0].plot(df_metrics['timestamp'], vwap, 
                 label='VWAP', color='darkblue', linewidth=2)
    
    # Add moving average
    if len(vwap) > 3:
        ma = vwap.rolling(window=3, min_periods=1).mean()
        axes[0].plot(df_metrics['timestamp'], ma, 
                     label='MA(3)', color='orange', linewidth=1.5, alpha=0.7)
    
    axes[0].set_title('Volume Weighted Average Price (VWAP)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Price (USDT)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Volume delta
if 'volume_profile_volume_delta' in df_metrics.columns:
    delta = df_metrics['volume_profile_volume_delta']
    colors = ['green' if x >= 0 else 'red' for x in delta]
    axes[1].bar(df_metrics['timestamp'], delta, color=colors, alpha=0.6, width=0.002)
    axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1].set_title('Volume Delta (Buy Volume - Sell Volume)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Volume Delta', fontsize=12)
    axes[1].set_xlabel('Time', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Visualization 3: Liquidity Metrics

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Trade arrival rate
if 'liquidity_trade_count' in df_metrics.columns:
    # Compute trades per 5 seconds
    axes[0].plot(df_metrics['timestamp'], df_metrics['liquidity_trade_count'], 
                 label='Trade Count (5s window)', color='darkgreen', linewidth=2)
    axes[0].set_title('Trade Activity', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Trades per 5s', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Price volatility
if 'liquidity_price_std' in df_metrics.columns:
    axes[1].plot(df_metrics['timestamp'], df_metrics['liquidity_price_std'], 
                 label='Price Std Dev', color='darkred', linewidth=2)
    axes[1].set_title('Price Volatility (Standard Deviation)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Std Dev (USDT)', fontsize=12)
    axes[1].set_xlabel('Time', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Statistical Summary

In [ ]:
# Select key metrics for summary
key_metrics = [
    'order_flow_buy_count',
    'order_flow_sell_count',
    'order_flow_order_imbalance',
    'volume_profile_vwap',
    'volume_profile_volume_delta',
    'liquidity_trade_count',
    'liquidity_price_std',
]

# Filter to available metrics
available_metrics = [m for m in key_metrics if m in df_metrics.columns]

if available_metrics:
    summary = df_metrics[available_metrics].describe()
    print("\nKey Metrics Summary:")
    print("=" * 80)
    print(summary.round(4))
else:
    print("No metrics available for summary")

## Aggregated Metrics (Multi-Window Analysis)

In [ ]:
# Show aggregated statistics from different time windows
print("\nAggregated Metrics Across Time Windows:")
print("=" * 80)

sample_metrics = ['order_flow_buy_count', 'volume_profile_vwap']

for metric in sample_metrics:
    if metric in df_metrics.columns:
        print(f"\n{metric}:")
        all_stats = aggregator.get_all_window_statistics(metric)
        
        for window_size, stats in all_stats.items():
            if stats['count'] > 0:
                print(f"  {window_size}s window:")
                print(f"    Mean: {stats['mean']:.4f}, Std: {stats['std']:.4f}")
                print(f"    Min: {stats['min']:.4f}, Max: {stats['max']:.4f}")

## Export Data

In [ ]:
# Save metrics to CSV
output_file = f'realtime_metrics_{SYMBOL}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_metrics.to_csv(output_file, index=False)
print(f"✓ Metrics saved to: {output_file}")
print(f"  Rows: {len(df_metrics)}")
print(f"  Columns: {len(df_metrics.columns)}")

## Next Steps

**Experiment with:**
1. Different time windows and durations
2. Multiple symbols simultaneously
3. Custom analyzers for specific patterns
4. Alert rules based on metrics

**See also:**
- `examples/simple_streaming_example.py` - Basic Python script
- `examples/realtime_monitoring.py` - CLI dashboard
- `README_STREAMING.md` - Complete documentation